# Convert Gemini Model Predictions to Annotation Spans

This notebook ingests pre-annotated data from Gemini (TSV) and aligns label predictions with character spans. 

Then it formats them to fit the Label Studio format.

## Environment Setup & Imports

In [1]:
import pandas as pd
import ast
import re
import uuid
import json

In [2]:
# Paths
GEMINI_ANNOTATIONS_INPUT_PATH = "../data/input/gemini_pre_annotated.tsv"
GEMINI_ANNOTATIONS_OUTPUT_PATH = "../data/output/pre-annotations-gemini.json"
GEMINI_ANNOTATIONS_OUTPUT_PATH_V2 = "../data/output/pre-annotations-gemini-v2.json"

## Load Dataset

In [3]:
predictions = pd.read_csv(GEMINI_ANNOTATIONS_INPUT_PATH, sep='\t')

In [4]:
# Since metadata is stored inside a stringified dictionary within the `data` column, we extract the unique ID (note_id) and the raw string content (note_content).

predictions['note_id'] = predictions['data'].apply(lambda x: ast.literal_eval(x)['id'])
predictions['note_content'] = predictions['data'].apply(lambda x: ast.literal_eval(x)['note_content'])
predictions.head()

,data,predictions,gemini_predictions,note_id,note_content
0,{'note_content': '[Category: Rents] Telephone ...,"[{'model_version': 'pre_annotated', 'score': 1...","{ ""labels"": [ { ""text"": ""Telephone line just r...",4d1f84da-85f0-9837-33ac-bdc3ae96fcba,[Category: Rents] Telephone line just ringing
1,{'note_content': '[Category: tenureManagement]...,"[{'model_version': 'pre_annotated', 'score': 1...","{ ""labels"": [ { ""text"": ""tenant"", ""label"": ""Pe...",685524bb-94d4-4f08-8669-0e78df744c0c,[Category: tenureManagement] Welfare check and...
2,{'note_content': '[Category: estateManagement]...,"[{'model_version': 'pre_annotated', 'score': 1...","{ ""labels"": [ { ""text"": ""tenant"", ""label"": ""Pe...",b7b3fb41-80b7-4d97-b8f4-0e0a7156ab79,[Category: estateManagement] arson attack lett...
3,{'note_content': '[Category: repairs] Repair i...,"[{'model_version': 'pre_annotated', 'score': 1...","{ ""labels"": [ { ""text"": ""tenant in number 5"", ...",918721e6-ccf6-445c-8cfc-658826b245cd,[Category: repairs] Repair issue Sent further ...
4,{'note_content': '[Category: Tenancy Managemen...,"[{'model_version': 'pre_annotated', 'score': 1...","{ ""labels"": [ { ""text"": ""Ms Aktar"", ""label"": ""...",5cac962d-dfc9-72d1-175a-612d7462cd4d,[Category: Tenancy Management] Dear Uju\n\nMs ...


## Data Validation

Verify that all generated values in the `gemini_predictions` column are valid JSON. Sometimes the model responds with something along the lines of "sorry, I can't do that." instead of the JSON we want, so this will catch that.

In [5]:
def _is_valid_json(x):
    try:
        json.loads(x)
        return True
    except (ValueError, TypeError):
        return False

mask = predictions['gemini_predictions'].apply(lambda x: not _is_valid_json(x))
display(predictions[mask])

,data,predictions,gemini_predictions,note_id,note_content


## Map Predictions to Character Offsets

Matches the raw predicted text segments to their character starting and ending indices using a sequential left-to-right regex scan. 

This helps when the same word/words are repeated in the note (e.g. 'tenant' may be mentioned multiple times). 

Without the left-to-right scan, the matching might highlight the same word multiple times instead of finding different instances.

In [6]:
def import_gemini_predictions(row):
    identified_labels = []
    id_lookup = {}
    last_match_end = 0  # track position to find closest match left-to-right

    # Handle label predictions (additional needs and roles)
    label_predictions = ast.literal_eval(row.gemini_predictions)['labels']

    for pred in label_predictions:
        # Find all matches and pick the one closest to (but not before) the last match
        matches = list(re.finditer(re.escape(pred['text']), row.note_content, re.IGNORECASE))
        if not matches:
            print(f"Warning: Could not find '{pred['text']}' in note_content for note_id {row.note_id}")
            continue
        match = min(matches, key=lambda m: abs(m.start() - last_match_end))
        last_match_end = match.end()
        
        # Create short unique ID & save to lookup
        entity_id = str(uuid.uuid4())[:8] 
        pred['id'] = entity_id
        id_lookup[pred['text']] = entity_id

        is_entity = pred['label'] in ('Person_Name', 'Person_Role', 'Person_Pronoun')
        identified_labels.append({
            "id": entity_id,
            "from_name": "entity_labels" if is_entity else "need_labels",
            "to_name": "text",
            "type": "labels",
            "value": {
                'start': match.start(),
                'end': match.end(),
                'text': pred['text'],
                'labels': [pred['label']]
            }
        })
    
    # Handle relation predictions
    link_predictions = ast.literal_eval(row.gemini_predictions)['links']
    for link in link_predictions:
        from_id = id_lookup.get(link['from'])
        to_id = id_lookup.get(link['to'])
        if not from_id or not to_id:
            print(f"Warning: Could not find entity IDs for link from '{link['from']}' to '{link['to']}' in note_id {row.note_id}")
            continue

        identified_labels.append({
            "from_id": from_id,
            "to_id": to_id,
            "type": "relation",
            "direction": "right"
        })

    return {
        "data": {
            "id": row.note_id,
            "note_content": row.note_content,
        },
        "predictions": [{
            "model_version": "pre_annotated",
            "score": 1.0,
            "result": identified_labels
        }]
    }

In [7]:
predictions_json = predictions.apply(import_gemini_predictions, axis=1)

## Export as Label Studio JSON

In [8]:
with open(GEMINI_ANNOTATIONS_OUTPUT_PATH, 'w') as f:
    json.dump(predictions_json.tolist(), f, indent=4)

## Fix: Relabel Person_Role -> Person_Name using spaCy

Gemini was not instructed to use `Person_Name` in the initial prompts, so all names were tagged as `Person_Role`.

This cell runs spaCy NER on each note and upgrades any `Person_Role` span that spaCy identifies as a `PERSON` entity to `Person_Name`.

In [9]:
import spacy
import copy

nlp = spacy.load('en_core_web_lg')

def relabel_person_names(task):
    task = copy.deepcopy(task)
    note_content = task['data']['note_content']
    doc = nlp(note_content)

    # Build set of (start, end) character offsets spaCy identifies as PERSON
    spacy_persons = {(ent.start_char, ent.end_char) for ent in doc.ents if ent.label_ == 'PERSON'}

    for item in task['predictions'][0]['result']:
        if item.get('type') != 'labels':
            continue # Skip relation links
        if item.get('from_name') != 'entity_labels':
            continue # Skip additional needs labels
        if item['value']['labels'] != ['Person_Role']:
            continue # Skip pronouns

        start, end = item['value']['start'], item['value']['end']

        # Compare with spaCy PERSON offsets to see if this span is actually a name
        is_name = any(
            s < end and e > start
            for s, e in spacy_persons
        )

        if is_name:
            item['value']['labels'] = ['Person_Name'] # If is a name, change label
    return task

predictions_v3 = [relabel_person_names(t) for t in predictions_json.tolist()]

# Sanity check
role_count = sum(1 for t in predictions_v3 for r in t['predictions'][0]['result'] if r.get('type') == 'labels' and r.get('from_name') == 'entity_labels' and r['value']['labels'] == ['Person_Role'])
name_count = sum(1 for t in predictions_v3 for r in t['predictions'][0]['result'] if r.get('type') == 'labels' and r.get('from_name') == 'entity_labels' and r['value']['labels'] == ['Person_Name'])
print(f'Person_Role: {role_count} | Person_Name: {name_count}')

Person_Role: 3470 | Person_Name: 1689


In [10]:
with open(GEMINI_ANNOTATIONS_INPUT_PATH, 'w') as f:
    json.dump(predictions_v3, f, indent=4)